### Generative Model Memory Sizes
Loads each generative model with HuggingFace transformers and measures GPU memory used.

Note: Transformers reflects actual model memory (allocates as needed), while vLLM reserves a fixed amount upfront for its PagedAttention KV cache.

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import gc

import torch
from transformers import AutoModelForCausalLM
from gptqmodel import GPTQModel
from vllm import LLM

WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0+5bb19ac
Transformers : 5.5.4
Torch        : 2.10.0+cu128
Triton       : 3.6.0


INFO  Loader: Auto dtype (native float16): `torch.float16`                     
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: batch_size.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: block_name_to_quantize.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: cache_block_outputs.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: dataset.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: exllama_config.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: max_input_length.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: model_seqlen.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: module_name_preceding_first_block.
INFO  QuantizeConfig: Ignoring unknown parameter in the quantization configuration: modules_in_bloc

In [ ]:
base_dir = '/groups/chichengz/tnn/datasets/'

model_names = [
    # "Llama3.2-1B-Instruct",
    "Llama3.2-3B-Instruct",
    "Llama3.2-3B-Instruct-GPTQ"
    "Qwen2.5-3B-Instruct",
    "Qwen2.5-3B-Instruct-GPTQ-Int4",
    "Qwen2.5-7B-Instruct",
    "Qwen2.5-7B-Instruct-GPTQ-Int4"
    # "Llama3.3-70B-Instruct-GPTQ",
]

In [3]:
def get_gpu_memory_used(device=0):
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)

### Measure with HuggingFace Transformers

In [7]:
tf_results = []

for name in model_names:
    print(f"\n=== {name} ===")
    llm_dir = base_dir + name

    if "GPTQ" in name:
        llm_tf = GPTQModel.load(llm_dir, device="cuda:0")
    else:
        llm_tf = AutoModelForCausalLM.from_pretrained(
            llm_dir,
            device_map="cuda:0",
            trust_remote_code=True,
        )
        llm_tf.eval()

    mem_gb = get_gpu_memory_used()
    print(f'  memory: {mem_gb:.2f} GB')
    tf_results.append((name, mem_gb))

    del llm_tf
    gc.collect()
    torch.cuda.empty_cache()

print("\n=== Summary (transformers) ===")
print(f"{'model':<35} {'memory (GB)':>12}")
print("-" * 48)
for name, mem in tf_results:
    print(f"{name:<35} {mem:>12.2f}")


=== Llama3.2-3B-Instruct ===


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  memory: 6.37 GB

=== Llama3.2-3B-Instruct-GPTQ ===
from_quantized: adapter: None


  memory: 2.50 GB

=== Qwen2.5-3B-Instruct-GPTQ-Int4 ===
from_quantized: adapter: None


  memory: 2.44 GB

=== Qwen2.5-7B-Instruct-GPTQ-Int4 ===
from_quantized: adapter: None


  memory: 5.66 GB

=== Summary (transformers) ===
model                                memory (GB)
------------------------------------------------
Llama3.2-3B-Instruct                        6.37
Llama3.2-3B-Instruct-GPTQ                   2.50
Qwen2.5-3B-Instruct-GPTQ-Int4               2.44
Qwen2.5-7B-Instruct-GPTQ-Int4               5.66


### Measure with vLLM (optional)
vLLM pre-allocates KV cache, so memory usage depends on `gpu_memory_utilization`.

In [5]:
# vllm_results = []

# for name in model_names:
#     print(f"\n=== {name} ===")
#     llm_dir = base_dir + name

#     llm_vllm = LLM(
#         model=llm_dir,
#         tensor_parallel_size=1,
#         max_model_len=5000,
#         gpu_memory_utilization=0.25,
#         enforce_eager=True,
#         distributed_executor_backend=None,
#         disable_log_stats=True,
#         dtype="float16",
#         seed=0,
#     )

#     mem_gb = get_gpu_memory_used()
#     print(f'  memory: {mem_gb:.2f} GB')
#     vllm_results.append((name, mem_gb))

#     del llm_vllm
#     gc.collect()
#     torch.cuda.empty_cache()

# print("\n=== Summary (vLLM) ===")
# print(f"{'model':<35} {'memory (GB)':>12}")
# print("-" * 48)
# for name, mem in vllm_results:
#     print(f"{name:<35} {mem:>12.2f}")